# 01 — Data Understanding
## E-Commerce Risk & Demand Intelligence System

**Scope of this notebook:** exploration only.

This notebook inspects `nigeria_ecommerce_major_project_25000.csv` as-is. It does **not**:
- train any model,
- permanently clean or overwrite the source data,
- drop rows,
- engineer or encode features,
- assume what the cancellation target or feature set will be.

Any parsed/derived columns created below (e.g. a datetime version of `order_timestamp`) exist only in memory for this notebook and are not written back to the CSV.

Decisions about targets, leakage rules, and modeling strategy are owned by the project's senior ML engineer and are **not** made in this notebook. The final section lists open questions that must be resolved before any modeling work begins.


## 1. Imports and Data Load

We load the CSV using a project-relative path (no hard-coded absolute paths), so the notebook works whether it is run from the repository root or from within `notebook/`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

RANDOM_SEED = 42
DATA_FILENAME = "nigeria_ecommerce_major_project_25000.csv"

# Project-relative candidates: this notebook is expected at notebook/01_data_understanding.ipynb
# and the dataset at data/<DATA_FILENAME>. Jupyter's default working directory is the notebook's
# own folder, so we check both "run from repo root" and "run from notebook/" cases.
candidate_paths = [
    Path("data") / DATA_FILENAME,
    Path("..") / "data" / DATA_FILENAME,
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find '{DATA_FILENAME}' in any of: "
        f"{[str(p) for p in candidate_paths]}. "
        "Place the CSV in the project's data/ folder, or run this notebook "
        "from the repository root or from notebook/."
    )

print(f"Loading data from: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)
print("Load complete.")


## 2. Dataframe Shape

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


**Why this matters:** confirms the file loaded correctly and matches the expected row/column count documented for this dataset. A mismatch here (e.g. far fewer rows than expected) is an early signal of a truncated file, a wrong delimiter, or an encoding issue — all of which must be resolved before any further analysis is trusted.


## 3. First 10 Rows and a Fixed-Seed Random Sample

In [ ]:
display(df.head(10))


In [ ]:
display(df.sample(n=5, random_state=RANDOM_SEED))


**Why this matters:** `head()` shows the data in its natural order, which can hide problems that only appear later in the file (e.g. a batch of rows from a different source or time period). A fixed-seed random sample (`random_state=42`) gives a second, reproducible look at rows from anywhere in the file, and lets anyone re-running this notebook see exactly the same sample rows for discussion.


## 4. Column Names and Data Types

In [ ]:
dtypes_df = df.dtypes.rename("dtype").to_frame()
dtypes_df["column"] = dtypes_df.index
dtypes_df = dtypes_df[["column", "dtype"]].reset_index(drop=True)
display(dtypes_df)


**Why this matters:** pandas infers types on load, and it frequently gets this wrong for real-world CSVs — dates and timestamps load as plain strings (`object`), and numeric-looking IDs can load as integers when they should be treated as categorical labels. This table is the reference point for deciding, in a later and separate step, which columns need explicit conversion before any modeling. No conversions are applied here except the read-only datetime check in Section 9.


## 5. Missing Values — Count and Percentage per Column

In [ ]:
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

missing_summary = (
    pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_pct,
    })
    .sort_values("missing_count", ascending=False)
)

display(missing_summary)


**Why this matters:** missing-value patterns inform imputation strategy later, but more importantly they can reveal *why* a value is missing (e.g. a field that is only populated after delivery, which would make it unavailable at prediction time — a leakage concern, not just a missing-data concern). No rows or columns are dropped in this notebook; this is observation only.


## 6. Duplicate Rows and Duplicate `order_id` Values

In [ ]:
exact_duplicate_rows = df.duplicated(keep=False)
print(f"Exact duplicate rows (all columns identical): {exact_duplicate_rows.sum():,}")

if exact_duplicate_rows.sum() > 0:
    display(df[exact_duplicate_rows].sort_values(list(df.columns)).head(20))


In [ ]:
if "order_id" in df.columns:
    duplicate_order_id_mask = df["order_id"].duplicated(keep=False)
    n_duplicate_order_ids = df.loc[duplicate_order_id_mask, "order_id"].nunique()
    print(f"Rows sharing a duplicated order_id: {duplicate_order_id_mask.sum():,}")
    print(f"Distinct order_id values that repeat: {n_duplicate_order_ids:,}")

    if duplicate_order_id_mask.sum() > 0:
        display(df[duplicate_order_id_mask].sort_values("order_id").head(20))
else:
    print("Column 'order_id' not found in this dataset — cannot check order_id duplication.")


**Why this matters:** this is the identifier-integrity check that determines what a single row actually represents (see the open question in the final section). If `order_id` repeats, one row is *not* one order — it may be one order line/item, which changes how the cancellation and quantity prediction targets must be defined. Exact full-row duplicates are separately checked because they can indicate an accidental double-export or ETL bug rather than a legitimate repeated order.


## 7. Descriptive Statistics — Numerical Columns

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
print(f"Numeric columns detected: {list(numeric_df.columns)}")
display(numeric_df.describe().T)


**Why this matters:** min/max/percentiles surface implausible values early (e.g. zero or negative `quantity`/`unit_price`, a `discount_percent` outside 0–100). These are candidates for a future, explicitly-logged cleaning decision — not something to silently fix now. Large gaps between the 75th percentile and the max are also a first signal for outliers that need business judgment (legitimate bulk order vs. data error), not automatic removal.


## 8. `order_status` — Unique Values and Counts

In [ ]:
if "order_status" in df.columns:
    status_counts = df["order_status"].value_counts(dropna=False)
    status_pct = (df["order_status"].value_counts(normalize=True, dropna=False) * 100).round(2)

    status_summary = pd.DataFrame({
        "count": status_counts,
        "percent": status_pct,
    })
    display(status_summary)
else:
    print("Column 'order_status' not found in this dataset.")


**Why this matters:** this is the raw material for the cancellation target — but the target itself is **not** defined here. In particular, note whether a `Refunded` (or similarly named) category exists alongside `Cancelled` and `Delivered`. How refunded orders should be labeled is explicitly listed as an open question below and must be decided by the senior ML engineer before any target column is created.


## 9. `order_timestamp` — Safe Datetime Conversion (Read-Only Check)

In [ ]:
if "order_timestamp" in df.columns:
    parsed_timestamps = pd.to_datetime(df["order_timestamp"], errors="coerce")

    n_unparseable = parsed_timestamps.isna().sum() - df["order_timestamp"].isna().sum()
    n_unparseable = max(n_unparseable, 0)

    print(f"Original missing values in order_timestamp: {df['order_timestamp'].isna().sum():,}")
    print(f"Values that failed to parse as datetime: {n_unparseable:,}")
    print(f"Earliest timestamp: {parsed_timestamps.min()}")
    print(f"Latest timestamp:   {parsed_timestamps.max()}")

    # NOTE: parsed_timestamps is a local, in-memory Series used only for this check.
    # It is intentionally NOT written back into df, to keep this notebook exploration-only.
else:
    print("Column 'order_timestamp' not found in this dataset.")


**Why this matters:** `errors="coerce"` converts unparseable values to `NaT` instead of raising an error, so we can see *how many* timestamps are malformed without the notebook crashing or silently guessing a format. The date range tells us how much historical coverage exists, which is directly relevant to the later (separate) question of whether any time-based feature or a temporal train/test split is feasible. This parsed series is not merged into `df`; the source dataframe is left untouched.


## Questions to Resolve Before Modeling

These are open decisions, not conclusions from this notebook. They must be resolved by the project owner in consultation with the senior ML engineer before any feature engineering, target creation, or model training begins.

1. **What does one row represent?**
   Is each row a distinct order (one `order_id` per row), an order line/item, or a customer session event? This depends directly on the `order_id` duplication check in Section 6 and determines the correct unit of prediction for both the cancellation and quantity models.

2. **How should refunded orders be defined for the cancellation target?**
   If `order_status` includes a `Refunded` category (or equivalent) alongside `Cancelled` and `Delivered`, a deliberate decision is needed: treat refunds as cancellations, as a separate third class, as delivered, or exclude them from the classification dataset entirely. This must not default silently — see Section 8.

3. **Which features are genuinely available at the moment an order is placed?**
   For every column, the question is: *was this known when the order was created?* This must be answered column-by-column against the leakage rules already established for this project (excluding `order_status`, `actual_delivery_days`, `customer_rating`, and any quantity-derived monetary fields such as `gross_amount`, `discount_amount`, `total_amount`).

4. **Which fields are only known after an outcome happens?**
   Beyond the fields already flagged as forbidden, this dataset should be reviewed for any other post-outcome fields (e.g. delivery timing, customer feedback, payment confirmation events) that could leak information about the target if included as features.

No answers to these questions are assumed or implied elsewhere in this notebook.
